In [1]:
# from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import seaborn as sns
sns.set()
import gc
gc.collect()
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import set_matplotlib_formats
from IPython.display import clear_output
set_matplotlib_formats('retina')
#from tqdm import tqdm
#tqdm.pandas()
import numpy as np
from pyhive import presto
from datetime import datetime, timedelta
# from bson import ObjectId
from functools import reduce
from sklearn.cluster import KMeans
# from pymongo import MongoClient
import glob
import warnings
warnings.filterwarnings('ignore')
from datetime import date
import json
import re
#import dtale
#from h3 import h3
# import pandasql as ps
import datetime
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose

/var/folders/fb/2nwyf_mx609gfjxv_jjnb76h0000gn/T/ipykernel_74935/1857937564.py:12: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats('retina')


In [2]:
metabase_connection = presto.connect(
        # host='bi-trino-2.serving.data.production.internal',
#         host='bi-trino.serving.data.production.internal',
        #host='bi-presto.serving.data.production.internal',
        #host='prime-trino.serving.data.production.internal',
          host='presto-gateway.processing.data.production.internal',
         
#         host="presto.processing.yoda.run",
        # host="processing.processing.data.production.internal",


        port=80,
        protocol='http',
        catalog ='hive',
        username='abhishek.akki@rapido.bike')

In [355]:
test_caps=pd.read_clipboard()

In [357]:
control_caps=pd.read_clipboard()

In [359]:
df_test=test_caps[['captain_id']]
df_control=control_caps[['captain_id']]

In [360]:
test_caps=df_test.captain_id.to_list()

In [361]:
control_caps=control_caps.captain_id.to_list()

In [362]:
df_control.shape

(12255, 1)

In [363]:
df_test.shape

(12255, 1)

In [324]:
df_tmp = pd.read_sql(
    '''
        select name cohort, userid captain_id
        from canonical.user_selector
        where selectorid in ('6801dd36fb2ace0001059fa3', '6801dda64a7510000146406a')
        and yyyymmdd ='20250418'
    ''', metabase_connection)

In [ ]:
df_tmp

In [112]:
df_test

,captain_id
0,5d98541961bf5a14d093657a
1,637eed15cf35d13bc0f8a167
2,5e12df3389355c2289ab70c5
3,632950230a5820d8e4de1986
4,61c606afd82740ff791de67e
...,...
12250,62172503869e6e2794643f9b
12251,65ac9f8d7067061f167aaa9b
12252,61dfd80c043dbb24190bc3f3
12253,64607e74b3d0c2312020629a


In [122]:
q=f"""
select
distinct  yyyymmdd,epoch,
eventprops__eventprops__action as action, 
eventprops__eventprops__city  as city,
eventprops__eventprops__userid as userId,
eventprops__eventprops__type as type
from
canonical.iceberg_production_data_raw_clevertap_captain_clevertap_captain_events_v2_eventname_byob_screen_immutable
where yyyymmdd>='2025018'
and eventprops__eventprops__userid in {tuple(df_test.captain_id.unique())}
"""
byob=pd.read_sql(q,metabase_connection)

In [ ]:
byob

In [108]:
df_control[df_control.captain_id.isin(df_tmp[df_tmp['cohort']=='cheap_subs_delhi_control'].captain_id.unique())]

,captain_id
0,603ddd7e35f8d4948cd6d48b
1,64a511497d6273bafafc4b13
2,60fa68f9d417d4858e7de2f2
3,6199083d9ce5a37dc0302bd8
4,66841d23c84cd02f071f527d
...,...
12250,65dff19a12d186a9845e9794
12251,662c5d706af91284b2783238
12252,5e621f55d174835e0ef70fa6
12253,66bb7a9039a64e476d7ebe39


In [110]:
df_test[df_test.captain_id.isin(df_tmp[df_tmp['cohort']=='cheap_subs_delhi_control'].captain_id.unique())]

,captain_id
1,637eed15cf35d13bc0f8a167
2,5e12df3389355c2289ab70c5
6,63f5bffd18f04f48ad8b1bdd
7,5d883605af79442156708752
9,651c47e2e4de7a5bf26e110f
...,...
12245,635e727fa1fcde30643f3879
12246,6266b270e98a64308f967deb
12248,673869d5b7089c696b0380a6
12249,661575cf3c7e8a7faeb9c936


In [58]:
df_tmp.captain_id.nunique()

28877

,cohort,captain_id


In [364]:
test_visited  = pd.read_sql(
    f'''
    select event_props_user_id as captain_id,
        case
            when yyyymmdd between '20250414' and '20250420' then 'pre'
            when yyyymmdd between '20250421' and '20250427' then 'post'
        end as pre_post, 
        event_props_source, 
        yyyymmdd
    from
        clevertap.captain_subscription_screen_visited_immutable
    where yyyymmdd >= '20250414'
    and event_props_user_id in {tuple(test_caps)}
    ''', metabase_connection)
test_visited

KeyboardInterrupt: 

In [ ]:
control_visited  = pd.read_sql(
    f'''
    select event_props_user_id as captain_id,
        case
            when yyyymmdd between '20250414' and '20250420' then 'pre'
            when yyyymmdd between '20250421' and '20250427' then 'post'
        end as pre_post, 
        event_props_source, 
        yyyymmdd
    from
        clevertap.captain_subscription_screen_visited_immutable
    where yyyymmdd >= '20250414'
    and event_props_user_id in {tuple(control_caps)}
    ''', metabase_connection)
control_visited

In [129]:
control_visited.event_props_source.unique()

array(['em_screen', 'deeplink', 'progress_bar', 'ingress_banner',
       'ct_banner', 'pn', 'hamburger', 'whatsapp', 'on_duty_nudge'],
      dtype=object)

In [130]:
test_visited.event_props_source.unique()

array(['deeplink', 'em_screen', 'ingress_banner', 'progress_bar',
       'ct_banner', 'hamburger', 'pn', 'whatsapp'], dtype=object)

In [71]:
finalTest=pd.pivot_table(data=test_visited,index=['event_props_source'],columns=['pre_post'],aggfunc={'captain_id':'nunique'}).reset_index()

In [72]:
finalControl=pd.pivot_table(data=control_visited,index=['event_props_source'],columns=['pre_post'],aggfunc={'captain_id':'nunique'}).reset_index()

In [78]:
finalTest.columns=['source','post','pre']

In [79]:
finalControl.columns=['source','post','pre']

In [228]:
test_visited['flag']='Test'
control_visited['flag']='Control'

In [229]:
test_visited

,captain_id,pre_post,event_props_source,yyyymmdd,flag
0,61801278c7609bf9d24af025,pre,deeplink,20250416,Test
1,64a2cadf10bd781e1a2a7eec,pre,ingress_banner,20250416,Test
2,64b016d6ed9c323ece46b05f,pre,em_screen,20250416,Test
3,61ffc5bd725c4c5f18ae2d48,pre,em_screen,20250416,Test
4,5d89d4e0486b0b214778a561,pre,deeplink,20250416,Test
...,...,...,...,...,...
16635,678d99464fec9d8e468882f0,pre,deeplink,20250415,Test
16636,673cba95423d245066be9646,pre,deeplink,20250413,Test
16637,5dcc05ef0856f14618291c64,pre,deeplink,20250413,Test
16638,6078f90d79860e8241b527fa,pre,deeplink,20250413,Test


In [230]:
merged=pd.concat([test_visited,control_visited],axis=0)

In [231]:
v1=merged.pivot_table(index=['pre_post'],columns=['flag'], values='captain_id', aggfunc='nunique')

In [232]:
v1.to_clipboard()

In [365]:
q=f"""
 with tbl as (
    select captain_id,yyyymmdd,
    case
            when yyyymmdd between '20250413' and '20250419' then 'pre'
            when yyyymmdd between '20250421' and '20250427' then 'post'
    end as pre_post,
    count(distinct order_id) as numOrder
    from orders.order_logs_fact
    where  yyyymmdd between '20250413' and '20250426'
    and captain_id in {tuple(df_test.captain_id.unique())}
    group by 1,2,3
 )
 select captain_id,pre_post,
        sum(case when numOrder>0 then 1 else 0 end) as net_days,
        sum(numOrder)  as totalOrder
 from tbl 
 group by 1,2
"""
test_pre=pd.read_sql(q,metabase_connection)
q=f"""
 with tbl as (
    select captain_id,yyyymmdd,
    case
            when yyyymmdd between '20250413' and '20250419' then 'pre'
            when yyyymmdd between '20250421' and '20250427' then 'post'
    end as pre_post,
    count(distinct order_id) as numOrder
    from orders.order_logs_fact
    where  yyyymmdd between '20250413' and '20250426'
    and captain_id in {tuple(df_control.captain_id.unique())}
    group by 1,2,3
 )
 select captain_id,
        pre_post,
        sum(case when numOrder>0 then 1 else 0 end) as net_days,
        sum(numOrder)  as totalOrder
 from tbl 
 group by 1,2
"""
control_pre=pd.read_sql(q,metabase_connection)

In [302]:
test_net_days=pd.pivot_table(data=test_pre,index=['pre_post'],aggfunc={'captain_id':'nunique','net_days':'sum','totalOrder':'sum'}).reset_index()

In [303]:
control_net_days=pd.pivot_table(data=control_pre,index=['pre_post'],aggfunc={'captain_id':'nunique','net_days':'sum','totalOrder':'sum'}).reset_index()

In [304]:
test_net_days['net_days_normalized']=(test_net_days['net_days']/test_net_days['captain_id']).round(2)
test_net_days['order_normalized']=(test_net_days['totalOrder']/test_net_days['captain_id']).round(2)

In [308]:
control_net_days['net_days_normalized']=(control_net_days['net_days']/control_net_days['captain_id']).round(2)
control_net_days['order_normalized']=(control_net_days['totalOrder']/control_net_days['captain_id']).round(2)

In [311]:
control_net_days.to_clipboard()

In [312]:
test_net_days.to_clipboard()

In [165]:
control_net_days['net_days_normalized']=(control_net_days['net_days']/control_net_days['captain_id']).round(2)
control_net_days['order_normalized']=(control_net_days['totalOrder']/control_net_days['captain_id']).round(2)

In [239]:
test_net_days.to_clipboard()

In [40]:
post_purchase_funnel = pd.read_sql('''
    cm_report as (
        select captain_id userid, yyyymmdd,
            sum(gmv) as gmv, 
            sum(order_earnings) as order_earnings, 
            sum(cm) cm, 
            sum(take) take, 
            sum(subs_orders) as subs_orders,
            sum(subs_gmv) as subs_gmv,
            sum(subs_cm) as subs_cm,
            sum(subs_captain_payout) as subs_captain_payout,
            sum(subs_total_incentives) as subs_total_incentives
        from reports.sql_ingestion_captain_cm_amt_dist_view cmr
        inner join user_selectors on cmr.captain_id = user_selectors.userid
        group by 1,2
    )
    select 
        userid, 
        cohort,
        yyyymmdd, 
        subscriptionruleid, 
        rulename, 
        ruleamount,
        validity,
        cardinality(split(progress__orders, '}')) as num_orders_done,
        gmv, 
        order_earnings,
        cm,
        take,
        subs_orders,
        subs_gmv,
        subs_cm,
        subs_captain_payout,
        subs_total_incentives
    from canonical.iceberg_domain_captain_subscription_progress_snapshot
    inner join user_selectors using(userid)
    left join test_rules using(subscriptionruleid)
    left join cm_report using(userid, yyyymmdd)
    where yyyymmdd >= '20250419'
    ''', metabase_connection)

In [169]:
q=f"""
        select user_id as captain_id,
        case
            when yyyymmdd between '20250413' and '20250419' then 'pre'
            when yyyymmdd between '20250421' and '20250427' then 'post'
        end as pre_post,
        yyyymmdd,amount
    from  captain.captain_subscription_immutable
    where yyyymmdd >= '20250413'
    and user_id in {tuple(control_caps)}
"""
control_subs_bought=pd.read_sql(q,metabase_connection)

In [171]:
q=f"""
        select user_id as captain_id,
        case
            when yyyymmdd between '20250413' and '20250419' then 'pre'
            when yyyymmdd between '20250420' and '20250426' then 'post'
        end as pre_post,
        yyyymmdd,amount
    from  captain.captain_subscription_immutable
    where yyyymmdd >= '20250413'
    and user_id in {tuple(test_caps)}
"""
test_subs_bought=pd.read_sql(q,metabase_connection)

In [459]:
q=f"""
select userid captain_id,a.amount,a.consumed,b.ruleamount,json_extract_scalar(json_parse(b.validity),'$[0].type') as subsType,
json_extract_scalar(json_parse(b.validity),'$[0].maxValue') as value,excludeincentivetypes,
a.yyyymmdd,
 case
        when a.yyyymmdd between '20250413' and '20250419' then 'pre'
        when a.yyyymmdd between '20250421' and '20250427' then 'post'
 end as pre_post
from canonical.iceberg_domain_captain_subscription_progress_snapshot a
left join canonical.iceberg_domain_captain_subscription_rules_immutable b
on a.subscriptionruleid=b.id
where a.yyyymmdd >= '20250413'
and userid in {tuple(control_caps)}
"""
control_subs_all=pd.read_sql(q,metabase_connection)
q=f"""
select userid captain_id,a.amount,a.consumed,b.ruleamount,json_extract_scalar(json_parse(b.validity),'$[0].type') as subsType,
json_extract_scalar(json_parse(b.validity),'$[0].maxValue') as value,excludeincentivetypes,
a.yyyymmdd,
 case
        when a.yyyymmdd between '20250413' and '20250419' then 'pre'
        when a.yyyymmdd between '20250421' and '20250427' then 'post'
 end as pre_post
from canonical.iceberg_domain_captain_subscription_progress_snapshot a
left join canonical.iceberg_domain_captain_subscription_rules_immutable b
on a.subscriptionruleid=b.id
where a.yyyymmdd >= '20250413' 
and userid in {tuple(test_caps)}
"""
test_subs_all=pd.read_sql(q,metabase_connection)

KeyboardInterrupt: 

In [246]:
test_subs_bought['Flag']='Test'
control_subs_bought['Flag']='Control'

In [247]:
subs_bought=pd.concat([test_subs_bought,control_subs_bought],axis=0)

In [201]:
p1=pd.pivot_table(data=subs_bought,index=['pre_post'],columns=['Flag'],aggfunc={'captain_id':'nunique'})

In [204]:
v1=v1.reset_index()
p1=p1.reset_index()

In [206]:
v1.to_clipboard()

In [265]:
control_subs_all[control_subs_all.subsType=='unlimited']

,captain_id,amount,consumed,ruleamount,subsType,value,pre_post
0,638039d9827ed6620741b345,15.0,False,15.0,unlimited,None,post
1,6491647e19c42e3e05ef2f88,11.0,False,11.0,unlimited,None,pre
2,5d6f676d6012fb46f2f34c3d,0.0,False,0.0,unlimited,None,post
3,627559ddb3bf0bd629bb0420,25.0,False,25.0,unlimited,None,post
4,6672568e1393ae3b5bb7ddbb,15.0,False,15.0,unlimited,None,pre
...,...,...,...,...,...,...,...
2174,67ee5ededce5920d401b39fc,19.0,False,19.0,unlimited,None,post
2175,67cbe849c5b522a4735112ee,15.0,False,15.0,unlimited,None,post
2176,6632400b1b92405a3938adb0,15.0,False,15.0,unlimited,None,post
2177,66bfb027246d7e0ab6dcbbb2,15.0,False,15.0,unlimited,None,None


In [272]:
control_subs_all[control_subs_all.amount==15].excludeincentivetypes.unique()

array(['["daily"]'], dtype=object)

In [273]:
control_subs_all[control_subs_all.amount==11].excludeincentivetypes.unique()

array(['["daily"]', '["daily","weekly"]'], dtype=object)

In [209]:
df_test.shape

(12255, 1)

In [210]:
df_control.shape

(12255, 1)

In [213]:
df_test.sort_values(by='captain_id')

,captain_id
7485,5737dfb7ddbec227bf208b62
9770,573f29159b0ffc2836777422
9823,573f292a9b0ffc283677b512
5100,573f292c9b0ffc283677c229
8497,573f292e9b0ffc283677cba0
...,...
9813,67fb671ef37dbf5003cedfc4
3979,67fb6bc0e59429a50b9f6e65
4908,67fb7b3e3de6fb5cbe85bc2f
6428,67fbbcf6edbb78f01618dce1


In [277]:
test_subs_all.pivot_table(index=['pre_post'],columns=['ruleamount'],aggfunc={'captain_id':'nunique'})

captain_id                                                          \
ruleamount       0.0    3.0   6.0    11.0   15.0  19.0  22.0  25.0  29.0 33.0   
pre_post                                                                        
post            106.0  263.0  84.0  487.0   51.0  14.0   1.0  11.0   2.0  1.0   
pre              62.0    NaN   NaN   85.0  115.0  37.0  10.0  41.0  23.0  NaN   

                            
ruleamount  39.0 45.0 49.0  
pre_post                    
post         4.0  2.0  1.0  
pre         14.0  4.0  NaN

In [235]:
p1.to_clipboard()

In [242]:
control_subs_all.groupby(['pre_post','captain_id']).captain_id.count()

pre_post  captain_id              
post      57889262da9e9ff446245fc3    5
          5b29fc90b31f0734c2a7a042    1
          5b934f707228ca3709404056    1
          5ba762bf4ea2d26dc44ca5f6    1
          5c1b74b6afe73d05ead57f53    1
                                     ..
pre       67f9280f12a9bc7cbd180697    1
          67fb38456b55dfd1ba04d1f9    5
          67fb585c3de6fb6b6d77efdc    2
          67fb66c826d14d308ac2e1fc    7
          67fb8b5f6b55dfe76626353d    2
Name: captain_id, Length: 1019, dtype: int64

In [290]:
test_subs_all[test_subs_all.pre_post=='post'][['amount', 'ruleamount', 'subsType', 'value']].drop_duplicates().sort_values(by=['amount'])

,amount,ruleamount,subsType,value
351,0.0,0.0,earnings,10000.0
1257,0.0,0.0,unlimited,None
750,3.0,3.0,rides,1
7,6.0,6.0,rides,2
0,11.0,11.0,unlimited,None
53,15.0,15.0,unlimited,None
66,19.0,19.0,unlimited,None
39,22.0,22.0,unlimited,None
50,25.0,25.0,unlimited,None
43,29.0,29.0,unlimited,None


In [291]:
control_subs_all[control_subs_all.pre_post=='post'][['amount', 'ruleamount', 'subsType', 'value']].drop_duplicates().sort_values(by=['amount'])

,amount,ruleamount,subsType,value
565,0.00,0.0,unlimited,None
89,11.00,11.0,unlimited,None
13,15.00,15.0,unlimited,None
5,19.00,19.0,unlimited,None
1324,19.00,19.0,earnings,300.0
4,22.00,22.0,unlimited,None
0,25.00,25.0,unlimited,None
14,29.00,29.0,unlimited,None
35,29.00,29.0,earnings,500.0
931,29.00,29.0,earnings,400.0


In [283]:
test_subs_all.columns

Index(['captain_id', 'amount', 'consumed', 'ruleamount', 'subsType', 'value',
       'excludeincentivetypes', 'pre_post'],
      dtype='object')

,captain_id,amount,consumed,ruleamount,subsType,value,excludeincentivetypes,pre_post
0,5cbb439254bc7263ff4422c9,11.0,False,11.0,unlimited,None,"[""daily""]",post
3,5c29ec5f4a267149c761c3b2,11.0,False,11.0,unlimited,None,"[""daily""]",pre
5,61658141d323b7384c8e311d,11.0,False,11.0,unlimited,None,"[""daily""]",post
8,64701f3f4e4083781cf9d843,11.0,False,11.0,unlimited,None,"[""daily""]",post
10,67fa183849e85b9e7211b898,11.0,False,11.0,unlimited,None,"[""daily"",""weekly""]",pre
...,...,...,...,...,...,...,...,...
2836,5f43f652d358d1f80c5121c0,11.0,False,11.0,unlimited,None,"[""daily""]",post
2837,6593a313ec0c22029bf386ac,11.0,False,11.0,unlimited,None,"[""daily""]",post
2838,6708d7447067cdc6015d1f30,11.0,False,11.0,unlimited,None,"[""daily""]",post
2839,6322c8570c01016ace089ffb,11.0,False,11.0,unlimited,None,"[""daily""]",None


In [281]:
test_subs_all.subsType.unique()

array(['unlimited', 'rides', 'earnings'], dtype=object)

In [282]:
test_subs_all.value.unique()

array([None, '2', '10000.0', '1'], dtype=object)

In [390]:
test_subs_caps_id=test_subs_all[test_subs_all.captain_id.unique()]

SyntaxError: incomplete input (3418095209.py, line 1)

In [330]:
control_subs_caps_id=control_subs_all.captain_id.unique()

In [477]:
q=f"""
 with tbl as (
    select captain_id,yyyymmdd,
    case
            when yyyymmdd between '20250413' and '20250419' then 'pre'
            when yyyymmdd between '20250421' and '20250427' then 'post'
    end as pre_post,
    count(distinct order_id) as numOrder
    from orders.order_logs_fact
    where  yyyymmdd between '20250413' and '20250427'
    and captain_id in {tuple(tcid)}
    group by 1,2,3
 )
 select captain_id,yyyymmdd,pre_post,
        sum(case when numOrder>0 then 1 else 0 end) as net_days,
        sum(numOrder)  as totalOrder
 from tbl 
 group by 1,2,3
"""
test_pre=pd.read_sql(q,metabase_connection)
q=f"""
 with tbl as (
    select captain_id,yyyymmdd,
    case
            when yyyymmdd between '20250413' and '20250419' then 'pre'
            when yyyymmdd between '20250421' and '20250427' then 'post'
    end as pre_post,
    count(distinct order_id) as numOrder
    from orders.order_logs_fact
    where  yyyymmdd between '20250413' and '20250426'
    and captain_id in {tuple(ccid)}
    group by 1,2,3
 )
 select captain_id,
        pre_post,
        yyyymmdd,
        sum(case when numOrder>0 then 1 else 0 end) as net_days,
        sum(numOrder)  as totalOrder
 from tbl 
 group by 1,2,3
"""
control_pre=pd.read_sql(q,metabase_connection)

In [340]:
test_pre.pivot_table(index=['pre_post'],aggfunc={'captain_id':'nunique','net_days':'sum','totalOrder':'sum'})

,captain_id,net_days,totalOrder
pre_post,,,
post,724,2234,12167
pre,660,1828,8182


In [341]:
control_pre.pivot_table(index=['pre_post'],aggfunc={'captain_id':'nunique','net_days':'sum','totalOrder':'sum'})

,captain_id,net_days,totalOrder
pre_post,,,
post,635,2086,11657
pre,545,1455,6531


In [315]:
test_subs_all=test_subs_all[test_subs_all.ruleamount>0]

In [314]:
control_subs_all=control_subs_all[control_subs_all.ruleamount>0]

array([11., 29., 39., 25., 49.,  6., 22., 15., 19., 33.,  3., 45.])

In [328]:
control_subs_all.pivot_table(index=['pre_post'],columns=['subsType'],aggfunc={'captain_id':'nunique'})

captain_id          
subsType   earnings unlimited
pre_post                     
post           15.0     568.0
pre             NaN     314.0

In [343]:
test_subs_all.groupby.captain_id.nunique()

896

In [348]:
tcid=test_subs_all[test_subs_all.pre_post=='post'].captain_id.unique()

In [347]:
ccid=control_subs_all[control_subs_all.pre_post=='post'].captain_id.unique()

In [351]:
test_pre[test_pre.captain_id.isin(tcid)].pivot_table(index=['pre_post'],aggfunc={'captain_id':'nunique','net_days':'sum','totalOrder':'sum'}).to_clipboard()

In [352]:
control_pre[control_pre.captain_id.isin(ccid)].pivot_table(index=['pre_post'],aggfunc={'captain_id':'nunique','net_days':'sum','totalOrder':'sum'}).to_clipboard()

In [371]:
control_subs_all.yyyymmdd.unique()

array(['20250419', '20250421', '20250420', '20250418', '20250417',
       '20250426', '20250424', '20250423', '20250416', '20250422',
       '20250425', '20250427'], dtype=object)

In [374]:
test_subs_all.yyyymmdd.unique()

array(['20250421', '20250417', '20250416', '20250424', '20250423',
       '20250420', '20250427', '20250425', '20250419', '20250422',
       '20250426', '20250418'], dtype=object)

In [381]:
control_subs_all=control_subs_all[control_subs_all.amount>0]

In [382]:
test_subs_all=test_subs_all[test_subs_all.amount>0]

In [383]:
test_subs_all.pivot_table(columns=['subsType'],index=['pre_post'],aggfunc={'captain_id':'nunique'})

captain_id          
subsType      rides unlimited
pre_post                     
post          351.0     524.0
pre             NaN     297.0

In [387]:
test_subs_all.pivot_table(columns=['amount'],index=['pre_post'],aggfunc={'captain_id':'nunique'}).to_clipboard()

In [388]:
control_subs_all.pivot_table(columns=['amount'],index=['pre_post'],aggfunc={'captain_id':'nunique'}).to_clipboard()

In [400]:
rides_plan=set(test_subs_all[test_subs_all.amount==3].captain_id)

In [408]:
test_subs_all.groupby(['captain_id', 'yyyymmdd'])['amount'].count().describe([x/10 for x in range(10)])

count    2053.000000
mean        1.255723
std         0.678529
min         1.000000
0%          1.000000
10%         1.000000
20%         1.000000
30%         1.000000
40%         1.000000
50%         1.000000
60%         1.000000
70%         1.000000
80%         1.000000
90%         2.000000
max         9.000000
Name: amount, dtype: float64

In [409]:
test_subs_all.groupby(['captain_id', 'yyyymmdd', 'amount'])['amount'].count().describe([x/10 for x in range(10)])

count    2172.000000
mean        1.186924
std         0.565436
min         1.000000
0%          1.000000
10%         1.000000
20%         1.000000
30%         1.000000
40%         1.000000
50%         1.000000
60%         1.000000
70%         1.000000
80%         1.000000
90%         2.000000
max         8.000000
Name: amount, dtype: float64

In [411]:
unlimited_plan=set(test_subs_all[(test_subs_all.pre_post=='post') & (test_subs_all.amount==11)].captain_id)

In [395]:
test_subs_all[(test_subs_all.pre_post=='post') & (test_subs_all.amount==11)].captain_id

,captain_id,amount,consumed,ruleamount,subsType,value,excludeincentivetypes,yyyymmdd,pre_post
0,61630e3a7fefc5aec58dc6a5,11.0,False,11.0,unlimited,None,"[""daily""]",20250421,post
3,61ffc5bd725c4c5f18ae2d48,11.0,False,11.0,unlimited,None,"[""daily""]",20250424,post
4,61658141d323b7384c8e311d,11.0,False,11.0,unlimited,None,"[""daily""]",20250423,post
5,67fa183849e85b9e7211b898,11.0,False,11.0,unlimited,None,"[""daily""]",20250423,post
6,612b177c890209d6765b94ad,11.0,False,11.0,unlimited,None,"[""daily""]",20250423,post
...,...,...,...,...,...,...,...,...,...
2836,645d08ad8e548142981877fd,11.0,False,11.0,unlimited,None,"[""daily""]",20250421,post
2840,6650bb6be190bd0726dbe12e,11.0,False,11.0,unlimited,None,"[""daily""]",20250425,post
2841,664c579f7095b03af7d7fe1a,11.0,False,11.0,unlimited,None,"[""daily""]",20250425,post
2842,6002e2f7857d1b101c2be076,11.0,False,11.0,unlimited,None,"[""daily""]",20250422,post


In [412]:
np.intersect1d(rides_plan,unlimited_plan)

array([], dtype=object)

In [414]:
test_subs_all[test_subs_all.pre_post == 'post'].groupby('captain_id')['amount'].nunique().value_counts()

amount
1    551
2    156
3     16
4      1
Name: count, dtype: int64

In [416]:
test_subs_all[test_subs_all.pre_post == 'post'].groupby(['captain_id', 'yyyymmdd'])['amount'].count().value_counts()

amount
1    1266
2     209
3      37
4      18
5       7
9       1
7       1
6       1
Name: count, dtype: int64

In [415]:
test_subs_all[test_subs_all.pre_post == 'post'].groupby(['captain_id', 'yyyymmdd'])['amount'].nunique().value_counts()

amount
1    1446
2      91
3       3
Name: count, dtype: int64

In [418]:
pd.crosstab(test_subs_all[test_subs_all.pre_post == 'post'].groupby(['captain_id', 'yyyymmdd'])['amount'].nunique(), test_subs_all[test_subs_all.pre_post == 'post'].groupby(['captain_id', 'yyyymmdd'])['amount'].count())

amount,1,2,3,4,5,6,7,9
amount,,,,,,,,
1,1266,147,16,11,5,1,0,0
2,0,62,20,5,2,0,1,1
3,0,0,1,2,0,0,0,0


In [420]:
t1=test_subs_all[(test_subs_all.pre_post == 'post') & (test_subs_all.amount==3)]

In [430]:
t2=test_subs_all[(test_subs_all.pre_post == 'post') & (test_subs_all.amount==11)]
t3=test_subs_all[(test_subs_all.pre_post == 'post') & (test_subs_all.amount==6)]

In [431]:
t1.merge(t2,how='inner',on='captain_id').captain_id.nunique()

113

In [688]:
t1.merge(t2,how='inner',on=['captain_id','yyyymmdd'])

,captain_id,amount_x,consumed_x,ruleamount_x,subsType_x,value_x,excludeincentivetypes_x,yyyymmdd,pre_post_x,amount_y,consumed_y,ruleamount_y,subsType_y,value_y,excludeincentivetypes_y,pre_post_y
0,67b984bfc51d5dd04eb31016,3.0,False,3.0,rides,1,"[""daily""]",20250424,post,11.0,False,11.0,unlimited,None,"[""daily""]",post
1,67b984bfc51d5dd04eb31016,3.0,False,3.0,rides,1,"[""daily""]",20250424,post,11.0,False,11.0,unlimited,None,"[""daily""]",post
2,67f16a263de6fb26c6484e3a,3.0,False,3.0,rides,1,"[""daily""]",20250422,post,11.0,False,11.0,unlimited,None,"[""daily""]",post
3,61bc8d267d49824be7bd03fe,3.0,False,3.0,rides,1,"[""daily""]",20250424,post,11.0,False,11.0,unlimited,None,"[""daily""]",post
4,64f57497a505a91cf187ef58,3.0,True,3.0,rides,1,"[""daily""]",20250427,post,11.0,False,11.0,unlimited,None,"[""daily""]",post
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,67f8bac1d37e028ce31d1eeb,3.0,True,3.0,rides,1,"[""daily""]",20250424,post,11.0,False,11.0,unlimited,None,"[""daily""]",post
96,67dd753e9724552beacfcca6,3.0,True,3.0,rides,1,"[""daily""]",20250426,post,11.0,False,11.0,unlimited,None,"[""daily""]",post
97,5f69b097f9451db7dfd3d4e4,3.0,False,3.0,rides,1,"[""daily""]",20250427,post,11.0,False,11.0,unlimited,None,"[""daily""]",post
98,6059f0bae757b2a0326e68a2,3.0,True,3.0,rides,1,"[""daily""]",20250426,post,11.0,False,11.0,unlimited,None,"[""daily""]",post


In [434]:
t2.merge(t3,how='inner',on='captain_id').captain_id.nunique()

55

In [435]:
t2.merge(t3,how='inner',on=['captain_id','yyyymmdd']).captain_id.nunique()

19

In [454]:
t3.merge(t1,how='inner',on='captain_id').captain_id.nunique()

37

In [455]:
t3.merge(t1,how='inner',on=['captain_id','yyyymmdd']).captain_id.nunique()

15

In [630]:
=

SyntaxError: invalid syntax (1763773627.py, line 1)

In [521]:
q=f"""
select userid captain_id,a.amount,a.consumed,b.ruleamount,json_extract_scalar(json_parse(b.validity),'$[0].type') as subsType,
json_extract_scalar(json_parse(b.validity),'$[0].maxValue') as value,excludeincentivetypes,
a.yyyymmdd,
a.updated_yyyymmdd,
case
        when a.yyyymmdd between '20250414' and '20250420' then 'pre'
        when a.yyyymmdd between '20250421' and '20250427' then 'post'
 end as pre_post,
cardinality(cast(json_parse(progress__orders) as array<json>)) as totalOrdes,
transform(cast(json_parse(progress__orders) as array<json>), x -> cast(json_extract_scalar(x, '$.commissionSaved') as decimal)) as commissionsaved,
transform(cast(json_parse(progress__orders) as array<json>), x -> cast(json_extract_scalar(x, '$.totalEarnings') as decimal)) as totalEarnings
from canonical.iceberg_domain_captain_subscription_progress_snapshot  a
left join canonical.iceberg_domain_captain_subscription_rules_immutable b
on a.subscriptionruleid=b.id
where a.yyyymmdd >= '20250414'
and userid in {tuple(control_caps)}
and amount>0
"""
control_subs_all=pd.read_sql(q,metabase_connection)
q=f"""
select userid captain_id,a.amount,a.consumed,b.ruleamount,json_extract_scalar(json_parse(b.validity),'$[0].type') as subsType,
json_extract_scalar(json_parse(b.validity),'$[0].maxValue') as value,excludeincentivetypes,
a.yyyymmdd,
a.updated_yyyymmdd,
 case
        when a.yyyymmdd between '20250414' and '20250420' then 'pre'
        when a.yyyymmdd between '20250421' and '20250427' then 'post'
 end as pre_post,
cardinality(cast(json_parse(progress__orders) as array<json>)) as totalOrdes,
transform(cast(json_parse(progress__orders) as array<json>), x -> cast(json_extract_scalar(x, '$.commissionSaved') as decimal)) as commissionsaved,
transform(cast(json_parse(progress__orders) as array<json>), x -> cast(json_extract_scalar(x, '$.totalEarnings') as decimal)) as totalEarnings
from canonical.iceberg_domain_captain_subscription_progress_snapshot a
left join canonical.iceberg_domain_captain_subscription_rules_immutable b
on a.subscriptionruleid=b.id
where a.yyyymmdd >= '20250414' 
and userid in {tuple(test_caps)}
and amount>0
"""
test_subs_all=pd.read_sql(q,metabase_connection)

In [573]:
tcid=test_subs_all[test_subs_all.pre_post=='post'].captain_id.unique()
ccid=control_subs_all[control_subs_all.pre_post=='post'].captain_id.unique()

## Performance

In [ ]:
q=f"""
with v0 as (
select userid captain_id,cast(a.amount as varchar) as amount,a.epoch,a.yyyymmdd
from canonical.iceberg_domain_captain_subscription_progress_snapshot a
left join canonical.iceberg_domain_captain_subscription_rules_immutable b
on a.subscriptionruleid=b.id
where a.yyyymmdd >= '20250421'
and userid in {tuple(test_caps)}
and amount!=0
)
select captain_id,listagg ( amount, '->' ) within group (order by epoch) as model,listagg ( yyyymmdd, '->' ) within group (order by yyyymmdd) as date
from v0
group by 1
"""
df=pd.read_sql(q,metabase_connection)

In [ ]:
df.groupby('model').captain_id.nunique().to_clipboard()

In [ ]:
tcid=test_subs_all[test_subs_all.pre_post=='post'].captain_id.unique()
ccid=control_subs_all[control_subs_all.pre_post=='post'].captain_id.unique()

### AO

In [576]:
q= f"""
with ao_table as (
      select lower(ao.event_props_ct_location_hex_8_city) city,
                        ao.profile_identity  captain_id,
                        ao.yyyymmdd yyyymmdd, 
    case
            when yyyymmdd between '20250414' and '20250420' then 'pre'
            when yyyymmdd between '20250421' and '20250427' then 'post'
      end as pre_post
      from clevertap.captain_app_launched_immutable ao
      where profile_identity in {tuple(tcid)} 
      and  yyyymmdd between '20250413' and '20250427'
)
select 
    ao_table.captain_id as captain_id,pre_post,yyyymmdd,count(ao_table.captain_id) as numAO
    from  ao_table as ao_table
group by 1,2,3
"""
test_ao=pd.read_sql(q,metabase_connection)
q=f"""
with ao_table as (
      select lower(ao.event_props_ct_location_hex_8_city) city,
                        ao.profile_identity  captain_id,
                        ao.yyyymmdd yyyymmdd, 
    case
            when yyyymmdd between '20250414' and '20250420' then 'pre'
            when yyyymmdd between '20250421' and '20250427' then 'post'
      end as pre_post
      from clevertap.captain_app_launched_immutable ao
      where profile_identity in {tuple(ccid)} 
      and  yyyymmdd between '20250413' and '20250427'
)
select 
    ao_table.captain_id as captain_id,pre_post,yyyymmdd,count(ao_table.captain_id) as numAO
    from  ao_table as ao_table
group by 1,2,3
"""
control_ao=pd.read_sql(q,metabase_connection)

In [578]:
control_ao[control_ao.numAO>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,547,513
yyyymmdd,2851,2345


In [588]:
test_ao.pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,680,624
yyyymmdd,3493,2674


### Net Days

In [582]:
q=f"""
 with tbl as (
    select captain_id,yyyymmdd,
    case
            when yyyymmdd between '20250414' and '20250420' then 'pre'
            when yyyymmdd between '20250421' and '20250427' then 'post'
    end as pre_post,
    count(distinct order_id) as numOrder
    from orders.order_logs_fact
    where  yyyymmdd between '20250413' and '20250427'
    and captain_id in {tuple(tcid)}
    group by 1,2,3
 )
 select captain_id,yyyymmdd,pre_post,
        sum(case when numOrder>0 then 1 else 0 end) as net_days,
        sum(numOrder)  as totalOrder
 from tbl 
 group by 1,2,3
"""
test_pre=pd.read_sql(q,metabase_connection)
q=f"""
 with tbl as (
    select captain_id,yyyymmdd,
    case
            when yyyymmdd between '20250414' and '20250420' then 'pre'
            when yyyymmdd between '20250421' and '20250427' then 'post'
    end as pre_post,
    count(distinct order_id) as numOrder
    from orders.order_logs_fact
    where  yyyymmdd between '20250413' and '20250426'
    and captain_id in {tuple(ccid)}
    group by 1,2,3
 )
 select captain_id,
        pre_post,
        yyyymmdd,
        sum(case when numOrder>0 then 1 else 0 end) as net_days,
        sum(numOrder)  as totalOrder
 from tbl 
 group by 1,2,3
"""
control_pre=pd.read_sql(q,metabase_connection)

In [628]:
control_pre[control_pre.net_days>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,556,458
yyyymmdd,1880,1438


In [629]:
test_pre[test_pre.net_days>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,685,508
yyyymmdd,2412,1504


### Gross Days

In [585]:
q=f"""
with dpi_table as (
select dpi.rider_id as captain_id,dpi.yyyymmdd as yyyymmdd,
case
        when yyyymmdd between '20250414' and '20250420' then 'pre'
        when yyyymmdd between '20250421' and '20250427' then 'post'
end as pre_post,count(case when dpi.event_type = 'rider_acknowledged' then order_id end) as countGrossPing
from orders.dispatch_propagation_immutable dpi
where  yyyymmdd between '20250414' and '20250427'
and rider_id in {tuple(tcid)}  
group by 1,2,3
)
select captain_id,yyyymmdd,pre_post, case when countGrossPing>0 then 1 else 0 end as grossDays
from dpi_table
"""
test_gross=pd.read_sql(q,metabase_connection)
q=f"""
with dpi_table as (
select dpi.rider_id as captain_id,dpi.yyyymmdd as yyyymmdd,
case
        when yyyymmdd between '20250414' and '20250420' then 'pre'
        when yyyymmdd between '20250421' and '20250427' then 'post'
end as pre_post,count(case when dpi.event_type = 'rider_acknowledged' then order_id end) as countGrossPing
from orders.dispatch_propagation_immutable dpi
where  yyyymmdd between '20250414' and '20250427'
and rider_id in {tuple(ccid)}  
group by 1,2,3
)
select captain_id,yyyymmdd,pre_post, case when countGrossPing>0 then 1 else 0 end as grossDays
from dpi_table
"""
control_gross=pd.read_sql(q,metabase_connection)

In [591]:
control_gross[control_gross.grossDays>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,582,516
yyyymmdd,2664,2056


In [592]:
test_gross[test_gross.grossDays>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,719,585
yyyymmdd,3186,2253


### Online

In [595]:
q=f"""
with ol_table as (
      select  distinct ol.event_props_user_id  captain_id,
                        ol.yyyymmdd yyyymmdd,
      case
        when yyyymmdd between '20250414' and '20250420' then 'pre'
        when yyyymmdd between '20250421' and '20250427' then 'post'
        end as pre_post
      from clevertap.captain_online_immutable ol
    where  yyyymmdd between '20250414' and '20250427'
    and event_props_user_id in {tuple(tcid)}  
)
select captain_id,yyyymmdd,pre_post,count(distinct captain_id) as onlineFlag
from ol_table
group by 1,2,3
"""
test_online=pd.read_sql(q,metabase_connection)
q=f"""
with ol_table as (
      select  distinct ol.event_props_user_id  captain_id,
                        ol.yyyymmdd yyyymmdd,
      case
        when yyyymmdd between '20250414' and '20250420' then 'pre'
        when yyyymmdd between '20250421' and '20250427' then 'post'
        end as pre_post
      from clevertap.captain_online_immutable ol
    where  yyyymmdd between '20250414' and '20250427'
    and event_props_user_id in {tuple(ccid)}  
)
select captain_id,yyyymmdd,pre_post,count(distinct captain_id) as onlineFlag
from ol_table
group by 1,2,3
"""
control_online=pd.read_sql(q,metabase_connection)

In [596]:
test_online[test_online.onlineFlag>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,704,589
yyyymmdd,3282,2350


In [597]:
control_online[control_online.onlineFlag>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,574,519
yyyymmdd,2727,2160


### Accepted Days

In [598]:
q=f"""
with dpi_table as (
select dpi.rider_id as captain_id,dpi.yyyymmdd as yyyymmdd,
case
        when yyyymmdd between '20250414' and '20250420' then 'pre'
        when yyyymmdd between '20250421' and '20250427' then 'post'
end as pre_post,count(case when dpi.event_type = 'rider_accepted' then order_id end) as acceptedPings
from orders.dispatch_propagation_immutable dpi
where  yyyymmdd between '20250414' and '20250427'
and rider_id in {tuple(tcid)}  
group by 1,2,3
)
select captain_id,yyyymmdd,pre_post, case when acceptedPings>0 then 1 else 0 end as acceptedDays
from dpi_table
"""
test_accepted=pd.read_sql(q,metabase_connection)
q=f"""
with dpi_table as (
select dpi.rider_id as captain_id,dpi.yyyymmdd as yyyymmdd,
case
        when yyyymmdd between '20250414' and '20250420' then 'pre'
        when yyyymmdd between '20250421' and '20250427' then 'post'
end as pre_post,count(case when dpi.event_type = 'rider_accepted' then order_id end) as acceptedPings
from orders.dispatch_propagation_immutable dpi
where  yyyymmdd between '20250414' and '20250427'
and rider_id in {tuple(ccid)}  
group by 1,2,3
)
select captain_id,yyyymmdd,pre_post, case when acceptedPings>0 then 1 else 0 end as acceptedDays
from dpi_table
"""
control_accepted=pd.read_sql(q,metabase_connection)

In [600]:
control_accepted[control_accepted.acceptedDays>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,574,464
yyyymmdd,2155,1484


In [601]:
test_accepted[test_accepted.acceptedDays>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,689,515
yyyymmdd,2470,1540


## Subs Level Details

In [602]:
q=f"""
with v0 as (
select userid captain_id,a.amount,a.consumed,b.ruleamount,json_extract_scalar(json_parse(b.validity),'$[0].type') as subsType,
json_extract_scalar(json_parse(b.validity),'$[0].maxValue') as value,excludeincentivetypes,
a.yyyymmdd,
 case
        when a.yyyymmdd between '20250414' and '20250420' then 'pre'
        when a.yyyymmdd between '20250421' and '20250427' then 'post'
 end as pre_post,
  transform(cast(json_parse(progress__orders) as array<json>), x -> json_extract_scalar(x, '$.orderId')) as orderId
from canonical.iceberg_domain_captain_subscription_progress_immutable  a
left join canonical.iceberg_domain_captain_subscription_rules_immutable b
on a.subscriptionruleid=b.id
where a.yyyymmdd >= '20250414'
and userid in {tuple(control_caps)}
and amount>0
),
v1 as ( 
select captain_id,amount,subsType,pre_post,value,yyyymmdd,t.orderId
from v0
cross join unnest(orderId) as t(orderId)
),
olf as(
  select order_id,yyyymmdd as orderDate
  from orders.order_logs_fact
  where order_id in (select orderid from v1)
  and order_status='dropped'
)
select captain_id,amount,subsType,value,pre_post,yyyymmdd,orderId, orderDate
from v1 a
left join olf
on orderId=order_id
"""
control_subs_only=pd.read_sql(q,metabase_connection)
q=f"""
with v0 as (select userid captain_id,a.amount,a.consumed,b.ruleamount,json_extract_scalar(json_parse(b.validity),'$[0].type') as subsType,
json_extract_scalar(json_parse(b.validity),'$[0].maxValue') as value,excludeincentivetypes,
a.yyyymmdd,
 case
        when a.yyyymmdd between '20250414' and '20250420' then 'pre'
        when a.yyyymmdd between '20250421' and '20250427' then 'post'
 end as pre_post,
 transform(cast(json_parse(progress__orders) as array<json>), x -> json_extract_scalar(x, '$.orderId')) as orderId
from canonical.iceberg_domain_captain_subscription_progress_snapshot a
left join canonical.iceberg_domain_captain_subscription_rules_immutable b
on a.subscriptionruleid=b.id
where a.yyyymmdd >= '20250414' 
and userid in {tuple(test_caps)}
and amount>0
),
v1 as ( 
select captain_id,amount,subsType,pre_post,value,yyyymmdd,t.orderId
from v0
cross join unnest(orderId) as t(orderId)
),
olf as(
  select order_id,yyyymmdd as orderDate
  from orders.order_logs_fact
  where order_id in (select orderid from v1)
  and order_status='dropped'
)
select captain_id,amount,subsType,value,pre_post,yyyymmdd,orderId, orderDate
from v1 a
left join olf
on orderId=order_id
"""
test_subs_only=pd.read_sql(q,metabase_connection)

In [603]:
test_subs_only

,captain_id,amount,subsType,value,pre_post,yyyymmdd,orderId,orderDate
0,61c9829e5379c0ce9d505c6f,11.0,unlimited,None,post,20250424,680a6c3fe37c88787d2d5a9c,20250424
1,61c9829e5379c0ce9d505c6f,11.0,unlimited,None,post,20250424,680ba361fcd3cf1151ef46f9,20250425
2,647d569ffd9f2afa434f8109,11.0,unlimited,None,post,20250425,680c7cb11024655a9c87d73b,20250426
3,647d569ffd9f2afa434f8109,11.0,unlimited,None,post,20250425,680c85755678c25d23171a31,20250426
4,647d569ffd9f2afa434f8109,11.0,unlimited,None,post,20250425,680c93938b3285533b072df5,20250426
...,...,...,...,...,...,...,...,...
8493,5ddd22f9a80df9312a1ce802,11.0,unlimited,None,post,20250423,680a127a5678c25d231020f2,20250424
8494,67fa52ba1856e01b8f3ab6a0,11.0,unlimited,None,post,20250424,680994d75678c25d230e4681,20250424
8495,67fa52ba1856e01b8f3ab6a0,11.0,unlimited,None,post,20250424,68099d586fd2d8338d279a40,20250424
8496,5ddd22f9a80df9312a1ce802,11.0,unlimited,None,post,20250423,6808d18ec733f155a907148f,20250423


In [608]:
control_subs_only.captain_id.nunique()

614

In [607]:
control_subs_all.pivot_table(columns=['pre_post'],aggfunc={'totalOrdes':'sum'})

pre_post,post,pre
totalOrdes,5883.0,2726.0


In [609]:
test_subs_all.pivot_table(columns=['pre_post'],aggfunc={'totalOrdes':'sum'})

pre_post,post,pre
totalOrdes,5883.0,2615.0


In [620]:
control_subs_only.pivot_table(columns=['pre_post'],aggfunc={'yyyymmdd':'nunique'})

pre_post,post,pre
orderId,5883,2726


In [622]:
test_subs_only.pivot_table(columns=['pre_post'],aggfunc={'orderId':'nunique'})

pre_post,post,pre
orderId,5883,2615


In [626]:
control_subs_only[['captain_id','pre_post','orderDate']].drop_duplicates().pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','orderDate':'count'})

pre_post,post,pre
captain_id,484,302
orderDate,1239,612


In [627]:
test_subs_only[['captain_id','pre_post','orderDate']].drop_duplicates().pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','orderDate':'count'})

pre_post,post,pre
captain_id,573,281
orderDate,1307,565


## Overall

### AO Overall

In [632]:
q= f"""
with ao_table as (
      select lower(ao.event_props_ct_location_hex_8_city) city,
                        ao.profile_identity  captain_id,
                        ao.yyyymmdd yyyymmdd, 
    case
            when yyyymmdd between '20250414' and '20250420' then 'pre'
            when yyyymmdd between '20250421' and '20250427' then 'post'
      end as pre_post
      from clevertap.captain_app_launched_immutable ao
      where profile_identity in {tuple(test_caps)} 
      and  yyyymmdd between '20250413' and '20250427'
)
select 
    ao_table.captain_id as captain_id,pre_post,yyyymmdd,count(ao_table.captain_id) as numAO
    from  ao_table as ao_table
group by 1,2,3
"""
test_all_ao=pd.read_sql(q,metabase_connection)
q=f"""
with ao_table as (
      select lower(ao.event_props_ct_location_hex_8_city) city,
                        ao.profile_identity  captain_id,
                        ao.yyyymmdd yyyymmdd, 
    case
            when yyyymmdd between '20250414' and '20250420' then 'pre'
            when yyyymmdd between '20250421' and '20250427' then 'post'
      end as pre_post
      from clevertap.captain_app_launched_immutable ao
      where profile_identity in {tuple(control_caps)} 
      and  yyyymmdd between '20250413' and '20250427'
)
select 
    ao_table.captain_id as captain_id,pre_post,yyyymmdd,count(ao_table.captain_id) as numAO
    from  ao_table as ao_table
group by 1,2,3
"""
control_all_ao=pd.read_sql(q,metabase_connection)

In [638]:
control_all_ao[control_all_ao.numAO>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,7224,7706
yyyymmdd,21707,22679


In [637]:
test_all_ao[test_all_ao.numAO>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,7046,7734
yyyymmdd,21258,22715


### Netdays Overall

In [633]:
q=f"""
 with tbl as (
    select captain_id,yyyymmdd,
    case
            when yyyymmdd between '20250414' and '20250420' then 'pre'
            when yyyymmdd between '20250421' and '20250427' then 'post'
    end as pre_post,
    count(distinct order_id) as numOrder
    from orders.order_logs_fact
    where  yyyymmdd between '20250413' and '20250427'
    and captain_id in {tuple(test_caps)}
    group by 1,2,3
 )
 select captain_id,yyyymmdd,pre_post,
        sum(case when numOrder>0 then 1 else 0 end) as net_days,
        sum(numOrder)  as totalOrder
 from tbl 
 group by 1,2,3
"""
test_all_pre=pd.read_sql(q,metabase_connection)
q=f"""
 with tbl as (
    select captain_id,yyyymmdd,
    case
            when yyyymmdd between '20250414' and '20250420' then 'pre'
            when yyyymmdd between '20250421' and '20250427' then 'post'
    end as pre_post,
    count(distinct order_id) as numOrder
    from orders.order_logs_fact
    where  yyyymmdd between '20250413' and '20250426'
    and captain_id in {tuple(control_caps)}
    group by 1,2,3
 )
 select captain_id,
        pre_post,
        yyyymmdd,
        sum(case when numOrder>0 then 1 else 0 end) as net_days,
        sum(numOrder)  as totalOrder
 from tbl 
 group by 1,2,3
"""
control_all_pre=pd.read_sql(q,metabase_connection)

In [639]:
control_all_pre[control_all_pre.net_days>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,3486,4042
yyyymmdd,8042,8855


In [640]:
test_all_pre[test_all_pre.net_days>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,3614,4008
yyyymmdd,8749,8717


### Gross Overall

In [634]:
q=f"""
with dpi_table as (
select dpi.rider_id as captain_id,dpi.yyyymmdd as yyyymmdd,
case
        when yyyymmdd between '20250414' and '20250420' then 'pre'
        when yyyymmdd between '20250421' and '20250427' then 'post'
end as pre_post,count(case when dpi.event_type = 'rider_acknowledged' then order_id end) as countGrossPing
from orders.dispatch_propagation_immutable dpi
where  yyyymmdd between '20250414' and '20250427'
and rider_id in {tuple(test_caps)}  
group by 1,2,3
)
select captain_id,yyyymmdd,pre_post, case when countGrossPing>0 then 1 else 0 end as grossDays
from dpi_table
"""
test_all_gross=pd.read_sql(q,metabase_connection)
q=f"""
with dpi_table as (
select dpi.rider_id as captain_id,dpi.yyyymmdd as yyyymmdd,
case
        when yyyymmdd between '20250414' and '20250420' then 'pre'
        when yyyymmdd between '20250421' and '20250427' then 'post'
end as pre_post,count(case when dpi.event_type = 'rider_acknowledged' then order_id end) as countGrossPing
from orders.dispatch_propagation_immutable dpi
where  yyyymmdd between '20250414' and '20250427'
and rider_id in {tuple(control_caps)}  
group by 1,2,3
)
select captain_id,yyyymmdd,pre_post, case when countGrossPing>0 then 1 else 0 end as grossDays
from dpi_table
"""
control_all_gross=pd.read_sql(q,metabase_connection)

In [642]:
control_all_gross[control_all_gross.grossDays>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,5488,6065
yyyymmdd,16106,16959


In [641]:
test_all_gross[test_all_gross.grossDays>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,5426,6060
yyyymmdd,15959,16721


### Online Caps

In [635]:
q=f"""
with ol_table as (
      select  distinct ol.event_props_user_id  captain_id,
                        ol.yyyymmdd yyyymmdd,
      case
        when yyyymmdd between '20250414' and '20250420' then 'pre'
        when yyyymmdd between '20250421' and '20250427' then 'post'
        end as pre_post
      from clevertap.captain_online_immutable ol
    where  yyyymmdd between '20250414' and '20250427'
    and event_props_user_id in {tuple(test_caps)}  
)
select captain_id,yyyymmdd,pre_post,count(distinct captain_id) as onlineFlag
from ol_table
group by 1,2,3
"""
test_all_online=pd.read_sql(q,metabase_connection)
q=f"""
with ol_table as (
      select  distinct ol.event_props_user_id  captain_id,
                        ol.yyyymmdd yyyymmdd,
      case
        when yyyymmdd between '20250414' and '20250420' then 'pre'
        when yyyymmdd between '20250421' and '20250427' then 'post'
        end as pre_post
      from clevertap.captain_online_immutable ol
    where  yyyymmdd between '20250414' and '20250427'
    and event_props_user_id in {tuple(control_caps)}  
)
select captain_id,yyyymmdd,pre_post,count(distinct captain_id) as onlineFlag
from ol_table
group by 1,2,3
"""
control_all_online=pd.read_sql(q,metabase_connection)

In [645]:
control_all_online[control_all_online.onlineFlag>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,5791,6378
yyyymmdd,17370,18431


In [646]:
test_all_online[test_all_online.onlineFlag>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,5705,6346
yyyymmdd,17263,18195


### Accepted Days

In [636]:
q=f"""
with dpi_table as (
select dpi.rider_id as captain_id,dpi.yyyymmdd as yyyymmdd,
case
        when yyyymmdd between '20250414' and '20250420' then 'pre'
        when yyyymmdd between '20250421' and '20250427' then 'post'
end as pre_post,count(case when dpi.event_type = 'rider_accepted' then order_id end) as acceptedPings
from orders.dispatch_propagation_immutable dpi
where  yyyymmdd between '20250414' and '20250427'
and rider_id in {tuple(test_caps)}  
group by 1,2,3
)
select captain_id,yyyymmdd,pre_post, case when acceptedPings>0 then 1 else 0 end as acceptedDays
from dpi_table
"""
test_all_accepted=pd.read_sql(q,metabase_connection)
q=f"""
with dpi_table as (
select dpi.rider_id as captain_id,dpi.yyyymmdd as yyyymmdd,
case
        when yyyymmdd between '20250414' and '20250420' then 'pre'
        when yyyymmdd between '20250421' and '20250427' then 'post'
end as pre_post,count(case when dpi.event_type = 'rider_accepted' then order_id end) as acceptedPings
from orders.dispatch_propagation_immutable dpi
where  yyyymmdd between '20250414' and '20250427'
and rider_id in {tuple(control_caps)}  
group by 1,2,3
)
select captain_id,yyyymmdd,pre_post, case when acceptedPings>0 then 1 else 0 end as acceptedDays
from dpi_table
"""
control_all_accepted=pd.read_sql(q,metabase_connection)

In [643]:
control_all_accepted[control_all_accepted.acceptedDays>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,3834,4212
yyyymmdd,9439,9419


In [644]:
test_all_accepted[test_all_accepted.acceptedDays>0].pivot_table(columns=['pre_post'],aggfunc={'captain_id':'nunique','yyyymmdd':'count'})

pre_post,post,pre
captain_id,3746,4170
yyyymmdd,9183,9223


In [651]:
test_subs_all[['captain_id','pre_post','yyyymmdd','subsType']].sort_values(by=['yyyymmdd']).drop_duplicates().pivot_table(index=['subsType'],columns=['pre_post'],aggfunc={"captain_id":'nunique'})

captain_id       
pre_post        post    pre
subsType                   
rides          351.0    NaN
unlimited      524.0  365.0

In [652]:
control_subs_all[['captain_id','pre_post','yyyymmdd','subsType']].sort_values(by=['yyyymmdd']).drop_duplicates().pivot_table(index=['subsType'],columns=['pre_post'],aggfunc={"captain_id":'nunique'})

captain_id       
pre_post        post    pre
subsType                   
earnings        17.0    NaN
unlimited      571.0  367.0

In [674]:
test_subs_all[test_subs_all.pre_post=='post'].pivot_table(index='amount',aggfunc={'totalOrdes':'sum'}).to_clipboard()

,captain_id,amount,consumed,ruleamount,subsType,value,excludeincentivetypes,yyyymmdd,updated_yyyymmdd,pre_post,totalOrdes,commissionsaved,totalEarnings
0,62f8782ccfaed18a09d97f6e,15.0,False,15.0,unlimited,None,"[""daily""]",20250421,20250421,post,NaN,None,None
1,67fb66c826d14d308ac2e1fc,19.0,False,19.0,unlimited,None,"[""daily"",""weekly""]",20250420,20250420,pre,NaN,None,None
2,66254538ec37c459ed8c0761,15.0,False,15.0,unlimited,None,"[""daily""]",20250421,20250421,post,NaN,None,None
3,5ffbbe1f87d741083a22a315,25.0,False,25.0,unlimited,None,"[""daily""]",20250422,20250422,post,NaN,None,None
4,628a48d6ef5c5726889795cc,11.0,False,11.0,unlimited,None,"[""daily""]",20250417,20250417,pre,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1922,5e5f53fd5a6831cf9973bd68,19.0,False,19.0,unlimited,None,"[""daily""]",20250427,20250427,post,2.0,"[5, 5]","[29, 31]"
1923,60c5ea2d558ac602106970f2,25.0,False,25.0,unlimited,None,"[""daily""]",20250424,20250424,post,NaN,None,None
1924,67514de064d2b71fb8e43850,15.0,False,15.0,unlimited,None,"[""daily""]",20250423,20250423,post,5.0,"[13, 11, 23, 7, 28]","[68, 56, 136, 35, 149]"
1925,66908428c8584785e8f642a4,22.0,False,22.0,unlimited,None,"[""daily""]",20250423,20250423,post,2.0,"[13, 4]","[76, 23]"


In [660]:
dft=test_subs_only.drop_duplicates()

In [659]:
dfc=control_subs_only.drop_duplicates()

In [665]:
dft

,captain_id,amount,subsType,value,pre_post,yyyymmdd,orderId,orderDate
0,61c9829e5379c0ce9d505c6f,11.0,unlimited,None,post,20250424,680a6c3fe37c88787d2d5a9c,20250424
1,61c9829e5379c0ce9d505c6f,11.0,unlimited,None,post,20250424,680ba361fcd3cf1151ef46f9,20250425
2,647d569ffd9f2afa434f8109,11.0,unlimited,None,post,20250425,680c7cb11024655a9c87d73b,20250426
3,647d569ffd9f2afa434f8109,11.0,unlimited,None,post,20250425,680c85755678c25d23171a31,20250426
4,647d569ffd9f2afa434f8109,11.0,unlimited,None,post,20250425,680c93938b3285533b072df5,20250426
...,...,...,...,...,...,...,...,...
8493,5ddd22f9a80df9312a1ce802,11.0,unlimited,None,post,20250423,680a127a5678c25d231020f2,20250424
8494,67fa52ba1856e01b8f3ab6a0,11.0,unlimited,None,post,20250424,680994d75678c25d230e4681,20250424
8495,67fa52ba1856e01b8f3ab6a0,11.0,unlimited,None,post,20250424,68099d586fd2d8338d279a40,20250424
8496,5ddd22f9a80df9312a1ce802,11.0,unlimited,None,post,20250423,6808d18ec733f155a907148f,20250423


In [684]:
dft[dft.pre_post=='post'][['captain_id','pre_post','amount','orderDate']].drop_duplicates().pivot_table(index=['pre_post'],aggfunc={'captain_id':'nunique'})

,captain_id
pre_post,
post,573


In [685]:
dft[dft.pre_post=='post'][['captain_id','pre_post','amount','orderDate']].drop_duplicates().pivot_table(index=['pre_post'],aggfunc={'orderDate':'count'})

,orderDate
pre_post,
post,1366


In [687]:
control_subs_all.pivot_table(index=['pre_post'],aggfunc={'captain_id':'nunique'})

,captain_id
pre_post,
post,585
pre,367


In [699]:
am3=test_subs_all[test_subs_all.amount==3].captain_id.unique()

In [695]:
df1=test_subs_all[test_subs_all.pre_post=='post'].pivot_table(index=['captain_id'],aggfunc={'amount':'nunique'}).reset_index()

In [694]:
df2=test_subs_all[test_subs_all.pre_post=='post'].pivot_table(index=['captain_id'],aggfunc={'amount':'nunique'}).reset_index()

In [696]:
df3=pd.merge(df1,df2,how='inner',on=['captain_id'])

In [701]:
df3=df3[df3.captain_id.isin(am3)]

In [702]:
df3

,captain_id,amount_x,amount_y
0,5a042fb983c486109539381d,2,2
2,5ba9f0206c55f962e6e30b64,1,1
4,5c085f3a395c1c0cd6d7878d,1,1
6,5c2a0c6f4a267149c761e3d6,1,1
7,5c44531f4a267149c7755485,1,1
...,...,...,...
710,67f4b3e70334193c660d46b2,1,1
712,67f68b4680636f4e02c34e63,3,3
715,67f8bac1d37e028ce31d1eeb,2,2
716,67f8ce173de6fb9e90a11d8a,1,1


In [704]:
test_subs_all[test_subs_all.captain_id=='5ba9f0206c55f962e6e30b64']

,captain_id,amount,consumed,ruleamount,subsType,value,excludeincentivetypes,yyyymmdd,updated_yyyymmdd,pre_post,totalOrdes,commissionsaved,totalEarnings
1602,5ba9f0206c55f962e6e30b64,3.0,False,3.0,rides,1,"[""daily""]",20250421,20250421,post,NaN,None,None
2567,5ba9f0206c55f962e6e30b64,3.0,False,3.0,rides,1,"[""daily""]",20250421,20250421,post,NaN,None,None


In [716]:
df1=test_subs_all[['captain_id','yyyymmdd','amount']].sort_values(by=['yyyymmdd']).groupby('captain_id')['amount'].agg(list).reset_index()

In [733]:
q=f"""
with v0 as (
select distinct userid captain_id,cast(a.amount as varchar) as amount,a.epoch,a.yyyymmdd
from canonical.iceberg_domain_captain_subscription_progress_snapshot a
left join canonical.iceberg_domain_captain_subscription_rules_snapshot b
on a.subscriptionruleid=b.id
where a.yyyymmdd >= '20250421'
and userid in {tuple(test_caps)}
and amount!=0
)
select captain_id,listagg ( amount, '->' ) within group (order by epoch) as model,listagg ( yyyymmdd, '->' ) within group (order by yyyymmdd) as date
from v0
group by 1
"""
df=pd.read_sql(q,metabase_connection)

In [736]:
df.groupby('model').captain_id.nunique().reset_index().to_clipboard()

In [746]:
morethanoncethresubs=test_subs_all[test_subs_all.amount==3].groupby(['captain_id','amount']).yyyymmdd.count().reset_index()

In [747]:
morethanoncethresubs=morethanoncethresubs[morethanoncethresubs.yyyymmdd>1]

In [753]:
morethanoncethresubs.to_csv('more_than_once_three_ruppee_subs.csv')

In [752]:
morethanoncethresubs.reset_index(drop=True,inplace=True)

In [754]:
morethanoncethresubs.to_clipboard()

In [757]:
test_mob=pd.read_clipboard()

In [760]:
test_mob=test_mob[['captain_id','mobile_number']]

In [761]:
morethanoncethresubs=morethanoncethresubs.merge(test_mob,on='captain_id',how='inner')

In [762]:
morethanoncethresubs

,captain_id,amount,yyyymmdd,mobile_number
0,5a042fb983c486109539381d,3.0,5,9711293936
1,5ba9f0206c55f962e6e30b64,3.0,2,8368742068
2,5c2a0c6f4a267149c761e3d6,3.0,2,8743803995
3,5caf50dd54bc7263ff33c5cd,3.0,2,8800367654
4,5cf28a06ca6e2921116f3d19,3.0,3,7983802391
...,...,...,...,...
101,67e649bfedbb783a7a0d0ad5,3.0,3,9990434146
102,67e95008edbb78a5a417a69d,3.0,3,9773651962
103,67efdd95dc1b954740d8e163,3.0,2,9306615221
104,67f34eeda053ad2739e8950e,3.0,2,9315332156


In [764]:
q=f"""
select captain_id,gig_segment_first
from (
     select captain_id,gig_segment_first,row_number() over(partition by captain_id order by run_date desc) as r
     from reports_internal.captain_gig_app_first_v7
     where captain_id in {tuple(test_caps)}
     and run_date between '20250410' and '20250427'
)
where r=1
"""
gig_status=pd.read_sql(q,metabase_connection)

In [767]:
morethanoncethresubs=morethanoncethresubs.merge(gig_status,on='captain_id',how='left')

In [769]:
morethanoncethresubs.to_clipboard()

In [ ]:
test_subs_all=test_subs_all.merge(gig)